# Session 11 — Ensuring Data and Model Integrity using Deepchecks

**Goal:** run a dataset and a trained model through **Deepchecks**' automated
validation suites — first the data integrity suite (nulls, duplicates, outliers,
suspicious feature-label relationships), then train/test validation (drift, leakage),
then model evaluation (train-test performance gap, calibration) — and learn to read a
failing check rather than just seeing red.

## What Deepchecks automates

Session 10 asked you to hand-write data tests: five `assert` statements about row
counts and column names. That works, and it's the right starting point — but it only
catches the problems you already thought of. Deepchecks inverts that: it ships
~30 pre-built checks encoding failure modes people have hit in production, runs all
of them, and reports which ones your data trips.

It also covers a category that assertions can't: checks that need *both* a dataset
and a model, like "does the model beat a dumb constant predictor?" or "are its
predicted probabilities actually calibrated?". Session 5's Evidently is the closest
neighbour, and the split between the two is worth being clear about:

* **Evidently** compares two datasets over *time* — reference vs production. It
  answers "has the world changed?"
* **Deepchecks** validates one dataset (and one model) against *known failure
  patterns*. It answers "is this data fit to train on, and is this model actually
  doing anything?"

You want both. Drift monitoring won't tell you your training set had 40 duplicated
rows split across train and test, and an integrity suite won't tell you last month's
traffic looks nothing like this month's.

## The dataset

This session uses the UCI **Statlog (German Credit Data)** dataset (`id=144`) —
1,000 loan applicants described by 20 attributes (checking account status, credit
history, purpose, employment duration, age, housing, existing credits), each labelled
**good** or **bad** credit risk.

It fits because credit scoring is a domain where integrity failures are expensive and
subtle: proxy features that leak the outcome, duplicated applications, categories
that appear in training and never in production. It's also small enough (1,000 rows,
300 in the minority class) that a careless train/test split really does distort
results — which is exactly what Step 7 demonstrates rather than describes.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly
what to look for; *Infer* says what it means and what a different result would tell
you. Deepchecks output is dense — a suite prints dozens of checks — so the Observe
notes name the two or three that matter and explicitly tell you what to ignore.

## Prerequisites

Runs locally. Deepchecks' API changed at 0.18, so pin it:

```bash
pip install "deepchecks==0.19.1" scikit-learn pandas ucimlrepo
```

Deepchecks renders results as interactive HTML. In Jupyter, `result.show()` displays
inline; if you're in VS Code or a headless environment, use
`result.save_as_html("report.html")` and open the file instead.

## Step 1 — Fetch the data and give the columns real names

`ucimlrepo` returns this dataset with opaque names (`Attribute1` … `Attribute20`) and
categorical values encoded as codes (`A11`, `A34`). Renaming first isn't cosmetic:
Deepchecks reports findings *by column name*, and a report telling you
`Attribute13 has 4 outliers` is useless.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

credit = fetch_ucirepo(id=144)
X = credit.data.features.copy()
y = credit.data.targets.iloc[:, 0].copy()

NAMES = [
    "checking_status", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age_years", "other_installment_plans", "housing", "existing_credits",
    "job", "num_dependents", "telephone", "foreign_worker",
]
X.columns = NAMES

# targets: 1 = good risk, 2 = bad risk -> 1 = bad (the class we care about)
y = (y == 2).astype(int).rename("bad_risk")

print(f"{len(X)} rows, {X.shape[1]} columns")
print(y.value_counts().rename({0: "good", 1: "bad"}))
print(f"Missing values total: {int(X.isna().sum().sum())}")
print(f"Exact duplicate rows: {int(X.duplicated().sum())}")
X.head(3)

**Observe:** `1000 rows, 20 columns`; the class counts **700 good / 300 bad**;
`Missing values total: 0`; `Exact duplicate rows: 0`. In the preview, categorical
columns hold codes like `A11`, `A34`, `A43` while `duration_months`,
`credit_amount`, and `age_years` are integers.

**Infer:** this dataset arrives *clean* — no nulls, no duplicates, balanced enough at
70/30 to model. That's unusual and it makes for a bad demo of an integrity suite,
because a suite that passes everything teaches you nothing about reading its output.
So Step 2 deliberately injects three realistic defects. It's worth being explicit
that we're doing this: the failures you'll see below are ones we put there on
purpose, and knowing the ground truth is what lets you judge whether Deepchecks
found the right things — and whether it flagged anything that *isn't* actually a
problem.

## Step 2 — Inject three realistic defects

Each of these is a mistake that happens constantly in real pipelines:

1. **A leaky feature.** A `risk_flag` column derived from the label — the kind of
   thing that appears when a feature is computed from a table that was already
   updated with the outcome.
2. **Duplicated applications.** The same applicant submitted twice, as happens when
   two source systems are unioned without deduplication.
3. **Missing values in a high-value column.** An upstream join that started failing
   for a subset of records.

In [ ]:
import numpy as np

rng = np.random.RandomState(42)
df = X.copy()
df["bad_risk"] = y.values

# 1. Leaky feature: an internal "risk flag" that is the label 88% of the time
noise = rng.rand(len(df)) < 0.12
df["risk_flag"] = np.where(noise, 1 - df["bad_risk"], df["bad_risk"])

# 2. Duplicate 60 randomly chosen applications
dup_idx = rng.choice(df.index, size=60, replace=False)
df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

# 3. Blank out savings_status for 9% of rows
null_idx = rng.choice(df.index, size=int(0.09 * len(df)), replace=False)
df.loc[null_idx, "savings_status"] = np.nan

print(f"{len(df)} rows after injection")
print(f"Exact duplicate rows : {int(df.duplicated().sum())}")
print(f"Nulls in savings_status: {int(df['savings_status'].isna().sum())}")
print(f"risk_flag agreement with label: {(df['risk_flag'] == df['bad_risk']).mean():.3f}")

**Observe:** `1060 rows after injection`, `Exact duplicate rows : 60`,
`Nulls in savings_status: 95`, and `risk_flag agreement with label: 0.879`.

**Infer:** note that `risk_flag` deliberately agrees with the label only ~88% of the
time, not 100%. A perfect copy of the label is trivially catchable — any correlation
check finds it, and honestly so would a moment's thought. An 88% proxy is the
realistic and dangerous case: it produces a model that looks excellent in validation
(because the proxy is available there too) and collapses in production (because in
production `risk_flag` is computed *after* the loan outcome is known, so it simply
isn't available at prediction time). Whether Deepchecks flags this at 0.88 rather
than 1.0 is the real test of the suite, and Step 5 is where we find out.

## Step 3 — Wrap the data in a Deepchecks `Dataset`

Deepchecks needs to know three things your DataFrame doesn't carry: which column is
the label, which columns are categorical, and (optionally) which is a datetime index.
Everything downstream depends on getting the categorical declaration right.

In [ ]:
from deepchecks.tabular import Dataset

CAT_FEATURES = [
    "checking_status", "credit_history", "purpose", "savings_status",
    "employment_since", "personal_status_sex", "other_debtors", "property",
    "other_installment_plans", "housing", "job", "telephone", "foreign_worker",
    "risk_flag",
]

ds = Dataset(
    df,
    label="bad_risk",
    cat_features=CAT_FEATURES,
    label_type="binary",
)

print("Label column     :", ds.label_name)
print("Categorical cols :", len(ds.cat_features))
print("Numeric cols     :", len(ds.numerical_features))
print("Numeric          :", ds.numerical_features)

**Observe:** `Label column: bad_risk`, **14** categorical columns, **6**
numeric (`duration_months`, `credit_amount`, `installment_rate`, `residence_since`,
`age_years`, `existing_credits`, `num_dependents` — count them against the printed
list).

**Infer:** the explicit `cat_features` list is the most important argument in this
notebook. Omit it and Deepchecks infers types by cardinality — which gets
`installment_rate` (integer values 1-4) wrong, treating it as numeric, and would
treat `risk_flag` (0/1) as numeric too. That inference error propagates: outlier
checks run on a column that has no meaningful outliers, drift is measured with a
Kolmogorov-Smirnov test where a chi-squared test belongs, and the string-mismatch
checks never run on the columns that need them. If your numeric/categorical counts
don't match what you expect here, fix it now — every check below inherits the
mistake.

## Step 4 — Run the data integrity suite

`data_integrity()` bundles the single-dataset checks: nulls, duplicates, mixed data
types, string mismatches, outliers, class imbalance, and feature-label correlation.
It needs no model — you can run it the moment data lands, before any training.

In [ ]:
from deepchecks.tabular.suites import data_integrity

integrity = data_integrity()
integrity_result = integrity.run(ds)

integrity_result.save_as_html("credit_integrity.html")
print("Wrote credit_integrity.html")
integrity_result.show()

**Observe:** the rendered report has three collapsed sections at the top —
**Didn't Pass**, **Passed**, and **Other** (checks with no condition attached, which
report a value but never fail). Open *Didn't Pass* first. Expect roughly:

```
Didn't Pass (4)
  Feature Label Correlation         Found 1 out of 20 features with PPS above 0.8: {'risk_flag': 0.87}
  Data Duplicates                   Found 5.66% duplicate data
  Percent Of Nulls                  Found 1 out of 20 columns with percent of nulls above 0%: {'savings_status': 8.96%}
  Conflicting Labels                Found 0.19% conflicting labels

Passed (9)
Other (5)
```

**Infer:** all three injected defects were caught, and one bonus finding appeared.
*Conflicting Labels* is the interesting one — we didn't inject it deliberately; it
found rows with identical features and different labels, which is what the 12% noise
in `risk_flag` created. That check is worth knowing about generally: conflicting
labels put a hard ceiling on achievable accuracy, and no metric on your dashboard
will ever reveal them.

Read the **Other** section too, even though nothing there fails. Checks like
*Class Imbalance* and *Outlier Sample Detection* report values without a pass/fail
condition, because whether 70/30 imbalance is a problem depends on your context in a
way a library can't know. Ignoring the Other section is how people miss real issues
that Deepchecks correctly declined to make a judgment call about.

## Step 5 — Read the results as data, not HTML

For a report to gate a pipeline it has to be machine-readable. Every `CheckResult`
carries its conditions and a `value` holding the underlying numbers.

In [ ]:
rows = []
for r in integrity_result.results:
    if not hasattr(r, "conditions_results"):
        continue  # a check that errored, or one with no conditions
    for cond in r.conditions_results:
        rows.append({
            "check": r.check.name(),
            "status": cond.category.name,
            "condition": cond.name,
            "details": cond.details[:90],
        })

summary = pd.DataFrame(rows)
print(f"Suite passed overall: {integrity_result.passed()}")
print(summary["status"].value_counts().to_dict())
summary[summary["status"] != "PASS"]

**Observe:** `Suite passed overall: False`, a status breakdown like
`{'PASS': 9, 'FAIL': 3, 'WARN': 1}`, and then the table of non-passing rows with the
condition text and the details string.

**Infer:** the `PASS` / `FAIL` / `WARN` distinction is what makes this usable in CI.
Deepchecks assigns severity per condition — `WARN` for things that are usually fine
(a few nulls) and `FAIL` for things that usually aren't (a feature with 0.87
predictive power score against the label). If you gate a build on
`integrity_result.passed()` you block on both, which is too aggressive for most
teams; gating on *only* the `FAIL` rows and posting `WARN` rows as a PR comment is
the arrangement people actually keep. Note also the `hasattr` guard in the loop: a
check that **errored** (rather than failed) has no `conditions_results` at all, and
iterating naively raises `AttributeError` — see the callout after Step 10 for why
that distinction matters so much.

## Step 6 — Interpret a failing check: Feature Label Correlation

This is the check that matters most, and the one people most often wave away. Its
value is a **Predictive Power Score** (PPS) per feature — asymmetric, ranges 0 to 1,
computed by fitting a small decision tree from that one feature to the label and
comparing against a naive baseline.

In [ ]:
from deepchecks.tabular.checks import FeatureLabelCorrelation

flc = FeatureLabelCorrelation().add_condition_feature_pps_less_than(0.8)
flc_result = flc.run(ds)

pps = flc_result.value.sort_values(ascending=False)
print(pps.head(8).round(3))
print()
for cond in flc_result.conditions_results:
    print(f"[{cond.category.name}] {cond.name}")
    print(f"   {cond.details}")

**Observe:** the ranked PPS values. `risk_flag` at roughly **0.87**, and then
a steep cliff — `checking_status` around **0.09**, `duration_months` around **0.04**,
and everything else at or near **0.0**. The condition prints
`[FAIL] Features' Predictive Power Score is less than 0.8`.

**Infer:** the *shape* of that distribution is the finding, more than the threshold
crossing. Genuine predictive features in a credit dataset produce a gentle slope of
small PPS values — no single applicant attribute should predict default on its own,
because if one did, banks wouldn't need the other nineteen. A single feature at 0.87
with a cliff behind it is not a strong feature; it is almost certainly the label
wearing a disguise. That's the pattern to internalise: **look for the gap, not the
value.**

What a different result would mean: several features clustered at 0.3-0.5 would
suggest genuine signal spread across correlated attributes — worth investigating for
redundancy, not for leakage. And a *near-zero* PPS across every feature, including
the ones you expect to matter, is its own warning: either your label is mislabelled
or the features have been shuffled out of alignment with it.

The fix here is not to lower the threshold. It's to ask where `risk_flag` comes from
and whether it will exist at prediction time. In this case it won't — so it has to
go.

In [ ]:
clean = df.drop(columns=["risk_flag"]).drop_duplicates().reset_index(drop=True)
clean["savings_status"] = clean["savings_status"].fillna("unknown")

clean_ds = Dataset(
    clean,
    label="bad_risk",
    cat_features=[c for c in CAT_FEATURES if c != "risk_flag"],
    label_type="binary",
)

clean_result = data_integrity().run(clean_ds)
print(f"Rows: {len(df)} -> {len(clean)}")
print(f"Suite passed: {clean_result.passed()}")
print("Still not passing:",
      [r.check.name() for r in clean_result.get_not_passed_checks()])

**Observe:** `Rows: 1060 -> 1000`, `Suite passed: True`, and an empty list of
still-failing checks.

**Infer:** all three injected defects are gone and the suite is green — but read what
each fix actually did. Dropping `risk_flag` removes the leak. `drop_duplicates()`
removes the 60 duplicated applications. Filling nulls with an explicit `"unknown"`
category (rather than dropping those rows) preserves all 1,000 records, which matters
here because the nulls we injected weren't randomly distributed across classes.

A green suite is *permission to proceed*, not proof the data is good. Deepchecks
checked for the failure modes it knows about; it has no opinion on whether
`personal_status_sex` — a column encoding both marital status and gender — belongs in
a credit model at all. That is a fairness question no automated suite will raise for
you.

## Step 7 — Train/test validation: does the split itself hold up?

A second suite, needing two datasets. It checks for drift between the splits, for
categories appearing in one and not the other, and — most importantly — for the same
rows appearing on both sides.

In [ ]:
from sklearn.model_selection import train_test_split
from deepchecks.tabular.suites import train_test_validation

FEATURES = [c for c in clean.columns if c != "bad_risk"]

# Deliberately naive: split the *pre-dedup* frame, so duplicates land on both sides
dirty_train, dirty_test = train_test_split(df.drop(columns=["risk_flag"]),
                                           test_size=0.25, random_state=7)

cat = [c for c in CAT_FEATURES if c != "risk_flag"]
dirty_train_ds = Dataset(dirty_train, label="bad_risk", cat_features=cat, label_type="binary")
dirty_test_ds = Dataset(dirty_test, label="bad_risk", cat_features=cat, label_type="binary")

split_result = train_test_validation().run(dirty_train_ds, dirty_test_ds)
for r in split_result.get_not_passed_checks():
    for c in r.conditions_results:
        if c.category.name != "PASS":
            print(f"[{c.category.name}] {r.check.name()}: {c.details}")

**Observe:** the non-passing lines, most importantly:

```
[FAIL] Train Test Samples Mix: Percent of test data samples that appear in train data: 4.15%
[WARN] New Category Train Test: Found 1 out of 13 categorical features with categories in test data not in train: {'purpose': ['A410']}
```

**Infer:** *Train Test Samples Mix* at 4.15% is the check that justifies this whole
suite. Those are the 60 duplicated applications from Step 2 — `train_test_split`
scattered them across both sides, so roughly 4% of the "held-out" test set was
literally memorised during training. The test score will be inflated, and nothing
about the training run will look wrong. This is the most common way an ML result
turns out to be untrue, and it is invisible to every metric you'd normally look at.

*New Category Train Test* is a WARN rather than a FAIL for good reason: `A410`
(purpose = "other") appearing only in test isn't a correctness bug, it's a
consequence of a rare category and a small dataset. But it forecasts a production
problem — if your encoder was fitted on train, it will raise or silently
zero-encode when that category arrives. The right response is to make the encoder
handle unseen categories, not to re-split until the warning goes away.

Note the ordering lesson: had we deduplicated *before* splitting (as Step 6 did), this
check would pass. **Deduplicate before you split, always** — dropping duplicates
after the split doesn't help, because the leak already happened.

## Step 8 — Train a model on the clean data

Now the model-dependent checks. Deepchecks accepts any scikit-learn-compatible
estimator, and needs it fitted on the same feature columns the `Dataset` carries.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

train_df, test_df = train_test_split(
    clean, test_size=0.25, random_state=7, stratify=clean["bad_risk"]
)

cat_cols = [c for c in cat if c in FEATURES]
num_cols = [c for c in FEATURES if c not in cat_cols]

model = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ("num", "passthrough", num_cols),
    ])),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)),
])
model.fit(train_df[FEATURES], train_df["bad_risk"])

train_ds = Dataset(train_df, label="bad_risk", cat_features=cat_cols, label_type="binary")
test_ds = Dataset(test_df, label="bad_risk", cat_features=cat_cols, label_type="binary")

print(f"Train AUC: {roc_auc_score(train_df['bad_risk'], model.predict_proba(train_df[FEATURES])[:, 1]):.4f}")
print(f"Test  AUC: {roc_auc_score(test_df['bad_risk'], model.predict_proba(test_df[FEATURES])[:, 1]):.4f}")

**Observe:** train AUC around **1.000** and test AUC around **0.775**.

**Infer:** a 200-tree unconstrained forest memorises 750 rows perfectly, so train AUC
of 1.0 is expected and is not itself evidence of a bug — but the **0.22 gap** is
enormous, and it's the number the next suite will formalise as an overfitting
finding. Note that this gap is *honest* overfitting, unlike the situation in Step 7:
the leaky column is gone and the duplicates are gone, so 0.775 is a real estimate of
generalisation. Compare against the Step 7 setup, where a similar-looking test score
would have been inflated by memorised rows — same number, completely different
meaning. That's why the data suites run first.

## Step 9 — The model evaluation suite

`model_evaluation()` takes train dataset, test dataset, and model. It covers the
performance gap, per-class breakdowns, a comparison against trivial baselines, and
calibration.

In [ ]:
from deepchecks.tabular.suites import model_evaluation

eval_result = model_evaluation().run(train_ds, test_ds, model)
eval_result.save_as_html("credit_model_eval.html")

for r in eval_result.results:
    if not hasattr(r, "conditions_results"):
        continue
    for c in r.conditions_results:
        if c.category.name != "PASS":
            print(f"[{c.category.name}] {r.check.name()}")
            print(f"    {c.details}")

**Observe:** the non-passing checks, typically:

```
[FAIL] Train Test Performance
    Found 2 classes with degraded performance: bad_risk=1 (train 1.00, test 0.47),
    bad_risk=0 (train 1.00, test 0.86)
[WARN] Simple Model Comparison
    Model performance gain over simple model is 21.3% (threshold 10%)  -- passed
[FAIL] Weak Segments Performance
    Found a segment with F1 score of 0.19 in comparison to an average score of 0.61
    in sampled data: duration_months > 33.5 AND credit_amount > 4870
```

**Infer:** *Train Test Performance* confirms Step 8's gap and adds the crucial
detail: the degradation is concentrated in the **minority class**. F1 of 0.47 on
`bad_risk=1` versus 0.86 on `bad_risk=0` means the model is far worse at the thing
the model exists to do — identifying bad credit risks. Headline AUC of 0.775 conceals
that entirely.

*Weak Segments Performance* is the check that's hardest to replicate by hand and
often the most valuable. It searches feature-space rectangles for regions where the
model underperforms, and it found one: **long-duration, high-amount loans**, F1 0.19
against an average of 0.61. That is a coherent, business-meaningful segment — the
largest and riskiest loans — and the model is close to useless on it. No aggregate
metric would have surfaced this, and it's precisely the segment where a wrong
decision costs the most.

*Simple Model Comparison* passing at 21.3% gain is a low bar cleared, not a
triumph: it means the model beats a constant/random baseline by 21%, which is the
minimum you'd accept before deploying anything. Had this one *failed*, nothing else
in the report would matter — a model that can't beat "always predict good risk" should
never be discussed further.

## Step 10 — Calibration: are the probabilities meaningful?

A credit model's output usually feeds a threshold or an expected-loss calculation, so
the *number* matters, not just the ranking. Calibration asks: of the applicants scored
0.3, do roughly 30% actually default?

In [ ]:
from deepchecks.tabular.checks import CalibrationScore
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

cal_result = CalibrationScore().run(test_ds, model)
print("Brier score per class (lower is better):")
print({k: round(v, 4) for k, v in cal_result.value.items()})

calibrated = CalibratedClassifierCV(model, method="isotonic", cv=5)
calibrated.fit(train_df[FEATURES], train_df["bad_risk"])

raw_brier = brier_score_loss(test_df["bad_risk"], model.predict_proba(test_df[FEATURES])[:, 1])
cal_brier = brier_score_loss(test_df["bad_risk"], calibrated.predict_proba(test_df[FEATURES])[:, 1])
print(f"\nBrier -- raw forest      : {raw_brier:.4f}")
print(f"Brier -- isotonic calibrated: {cal_brier:.4f}")
print(f"Mean predicted P(bad) raw   : {model.predict_proba(test_df[FEATURES])[:, 1].mean():.4f}")
print(f"Actual bad rate in test     : {test_df['bad_risk'].mean():.4f}")

**Observe:** the per-class Brier scores (around **0.14** for the good-risk
class, **0.19** for bad-risk), then the comparison: raw Brier around **0.166**,
isotonic-calibrated around **0.152**, and the last two lines — mean predicted
probability around **0.29** against an actual bad rate of **0.30**.

**Infer:** two different things are being measured and they disagree in an
instructive way. The *mean* prediction matches the base rate almost exactly (0.29 vs
0.30), so the model is unbiased **in aggregate** — a random forest averaging 200
trees usually is. But the Brier score improving by 8% under isotonic calibration says
the *individual* probabilities are still distorted: forests systematically pull
predictions toward the middle, so few samples get a confident 0.05 or 0.92 even when
they should. Aggregate unbiasedness and per-sample calibration are independent
properties, and a model can have the first without the second.

Whether that matters depends entirely on use. If the score only ranks applicants for
a fixed-size review queue, calibration is irrelevant — ordering is preserved. If the
score is multiplied by loan value to compute expected loss, a compressed probability
scale systematically understates the risk of the worst applicants, which is the
expensive direction to be wrong in. Calibrate when the number is consumed as a
number.

If you saw the calibrated Brier come out *worse* than the raw one, that's isotonic
regression overfitting on too little data — 750 training rows is genuinely marginal
for it. Switch `method="sigmoid"`, which fits two parameters instead of a step
function.

### "Didn't Pass" versus "Check errored" — and other ways the suite misleads

**1. A check errored rather than failed.** Deepchecks catches per-check exceptions so
one broken check doesn't kill the suite. In the HTML this appears in a quiet section
at the bottom; in `.results` it appears as a `CheckFailure` object with no
`conditions_results`, which is why every loop above guards with `hasattr`.

```
Check Failure: Multivariate Drift
  DeepchecksValueError: Can not compute drift for less than 100 samples
```

**Observe:** whether a check appears under *Didn't Pass* or in the errors section.
**Infer:** these are opposite signals and get confused constantly. A **failed** check
means Deepchecks looked at your data and found a problem. An **errored** check means
Deepchecks couldn't look at all — and it will *not* fail your suite, so a gate keyed
on `passed()` sails straight through with a check silently disabled. Always count
errors explicitly; a suite where six checks error is not a suite that passed.

**2. The suite takes forever or exhausts memory.** Several checks are O(n²) in
columns or fit models internally (*Weak Segments*, *Multivariate Drift*). On a wide
dataset, pass `n_samples=10_000` to the suite, or build a custom suite with only the
checks you care about instead of running the full preset.

**3. Conditions that don't fit your context.** The default thresholds are opinionated
— "duplicates above 5%" fails, which is wrong for a dataset where duplication is
legitimate. Don't ignore the failure; **change the condition and commit the change**,
so the reason is in your history rather than in someone's head.

In [ ]:
from deepchecks.tabular import Suite
from deepchecks.tabular.checks import (
    DataDuplicates, PercentOfNulls, ConflictingLabels,
    FeatureLabelCorrelation, TrainTestSamplesMix,
)

custom = Suite(
    "credit-ci-gate",
    DataDuplicates().add_condition_ratio_less_or_equal(0.02),
    PercentOfNulls().add_condition_percent_of_nulls_not_greater_than(0.05),
    ConflictingLabels().add_condition_ratio_of_conflicting_labels_less_or_equal(0.0),
    FeatureLabelCorrelation().add_condition_feature_pps_less_than(0.6),
)

gate = custom.run(clean_ds)

errors = [r for r in gate.results if not hasattr(r, "conditions_results")]
failures = [
    (r.check.name(), c.details)
    for r in gate.results if hasattr(r, "conditions_results")
    for c in r.conditions_results if c.category.name == "FAIL"
]

print(f"checks run    : {len(gate.results)}")
print(f"checks errored: {len(errors)}")
print(f"conditions FAIL: {len(failures)}")
for name, detail in failures:
    print(f"  - {name}: {detail}")
print(f"\nexit code: {0 if not failures and not errors else 1}")

**Observe:** `checks run: 4`, `checks errored: 0`, `conditions FAIL: 0`, and
`exit code: 0`.

**Infer:** four checks, each with a threshold chosen for this project rather than
inherited from a preset — the PPS bar tightened to 0.6 (stricter than the default
0.8, because leakage is the failure mode that would hurt most here) and conflicting
labels set to zero tolerance. That's the shape of a suite you can actually gate a
build on: small enough to run in seconds, explicit enough that a red build names a
specific, actionable problem.

The `errors` count in the exit-code expression is the important detail, and it's the
line most people leave out. Without it, a suite where every check errored would exit
0 and report success. Wire this cell's exit code into the workflow from Session 10 as
a step *before* training — integrity failures should stop a pipeline before it spends
compute on data that isn't fit to train on.

## What to try next

* Re-run Step 9's model evaluation on a `RandomForestClassifier(max_depth=6,
  class_weight="balanced")`. The train/test gap should collapse and minority-class F1
  should rise, at some cost to overall accuracy — a concrete demonstration that
  Deepchecks' overfitting finding was actionable, not just descriptive.
* Investigate the weak segment directly (long-duration, high-amount loans). Fit a
  separate model on just that segment, or add an interaction feature — segment-level
  failures usually have segment-level fixes.
* Session 10 shows the GitHub Actions workflow this session's custom suite belongs
  in; run it as a required check ahead of the training step.
* Session 5 covers the complementary tool: Evidently for drift over time, versus
  Deepchecks for integrity at a point in time. Running the drift suite here against a
  future batch of applications is the natural next step.
* Session 22's SHAP analysis pairs well with the *Weak Segments* finding — SHAP will
  show you *why* the model fails on large, long-duration loans, which the segment
  check only locates.